# Getting Started with Interactive Tables

## Load libraries and define custom functions

Firstly, make sure to have the prerequisite library installed in your Snowflake Notebook environment. 

To do this, click on **Packages** and add desirable package to the **Anaconda Packages** tab, but for this tutorial all the libraries comes pre-installed in the notebook environment.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time

from snowflake.snowpark.context import get_active_session
session = get_active_session()
cursor = session.connection.cursor()

def run_and_measure(count, mode):
    if mode == "std":
        query = """
                SELECT SearchEngineID, ClientIP, COUNT(*) AS c, SUM(IsRefresh), AVG(ResolutionWidth) FROM 
                BENCHMARK_FDN.HITS2_CSV
                WHERE SearchPhrase <> '' GROUP BY SearchEngineID, ClientIP ORDER BY c DESC LIMIT 10;
                """
        cursor.execute("USE WAREHOUSE wh")
    else:
        query = """
                SELECT SearchEngineID, ClientIP, COUNT(*) AS c, SUM(IsRefresh), AVG(ResolutionWidth) FROM 
                BENCHMARK_INTERACTIVE.CUSTOMERS
                WHERE SearchPhrase <> '' GROUP BY SearchEngineID, ClientIP ORDER BY c DESC LIMIT 10;
                """
        cursor.execute("USE WAREHOUSE interactive_demo_b")
    
    timings = []
    cursor.execute('ALTER SESSION SET USE_CACHED_RESULT = FALSE;')
    for i in range(count + 1):
        t0 = time.time()
        cursor.execute(query).fetchall()
        time_taken = time.time() - t0
        timings.append(time_taken)
            
    return timings[1:]
    
def plot_data(data, title, time_taken, color='#29B5E8'):
    # Separate titles and counts
    titles = [item[0] for item in data]
    counts = [item[1] for item in data]

    # Plot bar chart
    
    plt.figure(figsize=(12, 4))
    plt.bar(titles, counts, color=color)
    plt.xticks(rotation=45, ha='right')
    plt.ylabel("Counts")
    plt.xlabel("Title")
    plt.title(title)
    plt.text(0.5, 1.5, f'Time taken: {time_taken:.4f} seconds',
         ha='center', va='top',
         transform=plt.gca().transAxes,
         fontdict={'size': 16})
    #plt.tight_layout()
    plt.show()

## Set up role, warehouse, and database

Interactive Warehouses and Interactive Tables are now generally available (GA) and enabled by default on your account, so there's no need to check the Snowflake version or verify any account parameters.

Using a SQL cell, we'll set the active role and create the standard warehouse (`WH`), database (`MY_DEMO_DB`), and schemas used throughout this notebook. All statements use `IF NOT EXISTS`, so this cell is safe to re-run.

> **Note:** In a Snowflake Notebook, SQL and Python cells share the same session, so any `USE ROLE`, `USE DATABASE`, or `USE WAREHOUSE` statement you run in a SQL cell also applies to subsequent Python cells (and vice versa).

In [ ]:
USE ROLE ACCOUNTADMIN;

-- Create the compute and database objects used throughout this notebook (idempotent)
CREATE WAREHOUSE IF NOT EXISTS WH WITH WAREHOUSE_SIZE = 'X-SMALL';
CREATE DATABASE IF NOT EXISTS MY_DEMO_DB;
CREATE SCHEMA IF NOT EXISTS MY_DEMO_DB.BENCHMARK_FDN;
CREATE SCHEMA IF NOT EXISTS MY_DEMO_DB.BENCHMARK_INTERACTIVE;

USE WAREHOUSE WH;
USE DATABASE MY_DEMO_DB;

## Create an interactive warehouse & Turn it on

Next, let's create our `interactive_demo_b` warehouse and immediately turn it on:

In [ ]:
CREATE OR REPLACE INTERACTIVE WAREHOUSE interactive_demo_b
    WAREHOUSE_SIZE = 'XSMALL'
    MIN_CLUSTER_COUNT = 1
    MAX_CLUSTER_COUNT = 1
    COMMENT = 'Interactive warehouse demo';

## The Data

The next cell creates the `HITS2_CSV` table and loads it from the `synthetic_hits_data.csv` file bundled with this notebook. The load is **idempotent**: it checks whether the table already contains data and, if so, skips the load on subsequent runs.

> **Note:** The data is loaded from the bundled CSV using `pandas` + `write_pandas` (no external network access required). Make sure `synthetic_hits_data.csv` is added to this notebook's files.

In [ ]:
DB, SCHEMA, TABLE = "MY_DEMO_DB", "BENCHMARK_FDN", "HITS2_CSV"
FQ = f"{DB}.{SCHEMA}.{TABLE}"
CSV_FILE = "synthetic_hits_data.csv"  # bundled with this notebook

# Create the source table if it doesn't already exist
session.sql(f"""
CREATE TABLE IF NOT EXISTS {FQ} (
    EventDate DATE,
    CounterID INT,
    ClientIP STRING,
    SearchEngineID INT,
    SearchPhrase STRING,
    ResolutionWidth INT,
    Title STRING,
    IsRefresh INT,
    DontCountHits INT
)
""").collect()

# Idempotent load: only load when the table is empty
row_count = session.sql(f"SELECT COUNT(*) FROM {FQ}").collect()[0][0]
if row_count > 0:
    print(f"{FQ} already has {row_count:,} rows. Skipping data load.")
else:
    print(f"Loading data into {FQ} ...")
    pdf = pd.read_csv(CSV_FILE)
    pdf["EventDate"] = pd.to_datetime(pdf["EventDate"]).dt.date
    session.write_pandas(pdf, TABLE, database=DB, schema=SCHEMA, quote_identifiers=False)
    row_count = session.sql(f"SELECT COUNT(*) FROM {FQ}").collect()[0][0]
    print(f"Loaded {row_count:,} rows into {FQ}.")

In [ ]:
USE WAREHOUSE WH;
SELECT * FROM MY_DEMO_DB.BENCHMARK_FDN.HITS2_CSV;

## Create an interactive table

Now, we'll use the standard `WH` warehouse to efficiently create our new interactive `CUSTOMERS` table by copying all the data from the original standard table:

> **Note:** With zero-copy interactive analytics (public preview), creating an interactive table is now optional — an interactive warehouse can query the standard `HITS2_CSV` table directly. We still create an interactive table here to demonstrate that path and to enable a head-to-head performance comparison later in this notebook.

In [ ]:
-- Use a standard warehouse to build the interactive table's data
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE WH;
CREATE SCHEMA IF NOT EXISTS MY_DEMO_DB.BENCHMARK_INTERACTIVE;

CREATE OR REPLACE INTERACTIVE TABLE
  MY_DEMO_DB.BENCHMARK_INTERACTIVE.CUSTOMERS CLUSTER BY (ClientIP)
AS
  SELECT * FROM MY_DEMO_DB.BENCHMARK_FDN.HITS2_CSV;

## Attach interactive table to a warehouse

Next, we'll attach our interactive table to the warehouse, which pre-warms the data cache for optimal query performance:

> **Note:** `ADD TABLES` is a performance optimization, not a requirement. It proactively warms the warehouse's data cache so queries avoid a cold start. Any table you don't attach is still queryable and gets cached on demand the first time it's accessed. Proactive warming is currently limited to 10 tables.

In [ ]:
USE DATABASE MY_DEMO_DB;
ALTER WAREHOUSE interactive_demo_b ADD TABLES(BENCHMARK_INTERACTIVE.CUSTOMERS);

## Configure a fallback warehouse

Interactive warehouses are tuned for short, sub-second queries, so Snowflake fixes their statement timeout at a maximum of 5 seconds and automatically cancels any query that runs longer. To make sure an occasional heavy or ad-hoc query still completes instead of failing, you can designate a **fallback warehouse**: a standard warehouse that automatically re-runs any query that exceeds the 5-second timeout on the interactive warehouse.

This retry is transparent to the client (it behaves as an internal retry), so the query still returns its result. It keeps fast dashboard queries responsive while isolating them from the occasional long-running query.

We'll reuse the standard `WH` warehouse created earlier as the fallback, then confirm the setting via the `FALLBACK_WAREHOUSE` column.

> **Note:** The fallback is a **standard** warehouse (size it the same as or larger than the interactive warehouse) and must be started or set to auto-resume to accept retried queries. The querying role needs `USAGE` on both warehouses. To remove it later, run `ALTER WAREHOUSE interactive_demo_b UNSET FALLBACK_WAREHOUSE;`.

In [ ]:
ALTER WAREHOUSE interactive_demo_b SET FALLBACK_WAREHOUSE = WH;

SHOW WAREHOUSES LIKE 'interactive_demo_b';

## Run queries with interactive warehouse

Now, we'll run our first performance test on the interactive setup by executing a page-view query, timing its execution, and then plotting the results.

We'll start by activating the interactive warehouse and disabling the result cache using a SQL cell:

In [ ]:
USE WAREHOUSE interactive_demo_b;
USE DATABASE MY_DEMO_DB;
ALTER SESSION SET USE_CACHED_RESULT = FALSE;

Next, in a Python cell we'll run a query to find the top 10 most viewed pages for July 2013, measure how long it takes, and then plot the results and execution time:

In [ ]:
query = """
SELECT Title, COUNT(*) AS PageViews
FROM BENCHMARK_INTERACTIVE.CUSTOMERS
WHERE CounterID = 62
  AND EventDate >= '2013-07-01'
  AND EventDate <= '2013-07-31'
  AND DontCountHits = 0
  AND IsRefresh = 0
  AND Title <> ''
  AND REGEXP_LIKE(Title, '^[\\x00-\\x7F]+$')
  AND LENGTH(Title) < 20
GROUP BY Title
ORDER BY PageViews DESC
LIMIT 10;
"""

start_time = time.time()
result = cursor.execute(query).fetchall()
end_time = time.time()
time_taken = end_time - start_time

plot_data(result, "Page visit analysis (Interactive)", time_taken)

## Compare to a standard warehouse

To establish a performance baseline, we'll run an identical page-view query on a standard warehouse to measure and plot its results for comparison.

We'll start by preparing the session for a performance benchmark by selecting a standard `WH` warehouse, disabling the result cache, and setting the active database using a SQL cell:

In [ ]:
USE WAREHOUSE WH;
USE DATABASE MY_DEMO_DB;
ALTER SESSION SET USE_CACHED_RESULT = FALSE;

Here, in a Python cell we'll run a top 10 page views analysis by executing the query, measuring its performance, and immediately plotting the results and execution time:

In [ ]:
query = """
SELECT Title, COUNT(*) AS PageViews
FROM BENCHMARK_FDN.HITS2_CSV
WHERE CounterID = 62
  AND EventDate >= '2013-07-01'
  AND EventDate <= '2013-07-31'
  AND DontCountHits = 0
  AND IsRefresh = 0
  AND Title <> ''
  AND REGEXP_LIKE(Title, '^[\\x00-\\x7F]+$')
  AND LENGTH(Title) < 20
GROUP BY Title
ORDER BY PageViews DESC
LIMIT 10;
"""

start_time = time.time()
result = cursor.execute(query).fetchall()
end_time = time.time()
time_taken = end_time - start_time

plot_data(result, "Page visit analysis (Standard)", time_taken, '#5B5B5B')


## Run some queries concurrently

To directly compare performance, we'll benchmark both the interactive and standard warehouses over several runs and then plot their latencies side-by-side in a grouped bar chart:

In [ ]:
runs = 5

counts_iw = run_and_measure(runs,"iw")
print(counts_iw)

counts_std = run_and_measure(runs,"std")
print(counts_std)

titles = [f"R{i}" for i in range(1, len(counts_iw)+1)]

x = np.arange(len(titles))  # the label locations
width = 0.35  # bar width

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width/2, counts_std, width, label="Standard", color="#5B5B5B")
ax.bar(x + width/2, counts_iw, width, label="Interactive", color="#29B5E8")

ax.set_ylabel("Latency")
ax.set_xlabel("Query run")
ax.set_title("Standard vs Interactive warehouse")
ax.set_xticks(x)
ax.set_xticklabels(titles)
ax.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.15),
    ncol=2
)
plt.show()

